
# $t_a$ について $\tau_1^\theta$ を二通りに検算する

この notebook の目的は、現在採用している symplectic generators

$$
\pi=\langle x,y,z,w\rangle,\qquad
\zeta=[x,y][z,w]
$$

のもとで、KnotInfo の $a$-curve に沿う右 Dehn twist $t_a$ について

$$
\tau_1^\theta(t_a)
$$

を **二つの独立なルート**で計算し、一致することを確認することです。

- **Route B（$\pi$ への作用から）**  
  $t_a(x),t_a(y),t_a(z),t_a(w)$ を使い、
  $$
  \ell_2(t_a(g))
  =
  \tau_1^\theta(t_a)(|t_a(g)|)
  +(\Lambda^2|t_a|)\ell_2(g)
  $$
  から $\tau_1^\theta(t_a)$ を求める。

- **Route A（Dehn twist formula から）**  
  `kiyoh.pdf` の公式
  $$
  \tau_1^\theta(t_c)=-|c|\wedge\ell_2(c)
  $$
  に $c=a$ を代入する。

今回は $a=x$ なので Route A は特に簡単になり、最後には両方とも零行列になることを確認します。

> この notebook は検算過程を読むため、必要なコードだけを現在の notebook から抜き出した最小構成です。



## 0. Convention

自由群の生成元を

$$
x=x_1,\quad y=y_1,\quad z=x_2,\quad w=y_2
$$

とし、可換化を

$$
X,Y,Z,W
$$

と書きます。交点形式は

$$
X\cdot Y=1,\qquad Z\cdot W=1
$$

で、それ以外は $0$ です。

交換子は

$$
[u,v]=uvu^{-1}v^{-1}
$$

とし、境界語は

$$
\zeta=[x,y][z,w].
$$

$\Lambda^2H$ の基底順序は

$$
X\wedge Y,\ X\wedge Z,\ X\wedge W,\
Y\wedge Z,\ Y\wedge W,\ Z\wedge W.
$$


In [1]:
from sage.all import (
    FreeGroup, ZZ, QQ,
    vector, matrix,
    zero_matrix
)

%display latex

In [2]:
# 自由群と境界語

class fundamentalGroup:
    F = FreeGroup(4, 'x,y,z,w')
    x, y, z, w = F.generators()

    X, Y, Z, W = ~x, ~y, ~z, ~w
    BASIS = (x, y, z, w)

    @classmethod
    def comm(cls, u, v):
        if u.parent() != cls.F or v.parent() != cls.F:
            raise TypeError("u and v must be elements of fg.F")
        return u * v * (~u) * (~v)


fg = fundamentalGroup

fg.bnd = (
    fg.comm(fg.x, fg.y)
    * fg.comm(fg.z, fg.w)
)

fg.bnd

x*y*x^-1*y^-1*z*w*z^-1*w^-1


## 1. $t_a$ の $\pi$ への作用

現在の図から得た $a$-curve は

$$
a=x
$$

で、右 Dehn twist $t_a$ の自由群への作用は

$$
t_a(x)=x,\qquad
t_a(y)=yx,\qquad
t_a(z)=z,\qquad
t_a(w)=w.
$$

まずこれだけをコードにします。


In [3]:
def apply_automorphism(element, action):
    # action は {'x': image_of_x, ...} という辞書。
    # 指定されていない生成元は固定する。
    gen_names = {1: 'x', 2: 'y', 3: 'z', 4: 'w'}

    result = fg.F.one()

    for idx in element.Tietze():
        name = gen_names[abs(idx)]
        generator = fg.F.gen(abs(idx) - 1)
        image = action.get(name, generator)
        result *= image if idx > 0 else ~image

    return result


ACTION_a = {
    'y': fg.y * fg.x
}

def t_a(element):
    return apply_automorphism(element, ACTION_a)


images_a = [t_a(g) for g in fg.BASIS]

for name, image in zip(('x','y','z','w'), images_a):
    print(f"t_a({name}) = {image}")

t_a(x) = x
t_a(y) = y*x
t_a(z) = z
t_a(w) = w



期待される出力は

```text
t_a(x) = x
t_a(y) = y*x
t_a(z) = z
t_a(w) = w
```

です。

また $t_a\in\mathrm{Aut}_\partial(\pi)$ なので、境界語を固定する必要があります。


In [4]:
assert t_a(fg.bnd) == fg.bnd
print("t_a(bnd) == bnd :", t_a(fg.bnd) == fg.bnd)

t_a(bnd) == bnd : True



ここでは実際、

$$
[x,yx]=[x,y]
$$

なので

$$
t_a([x,y][z,w])=[x,y][z,w]
$$

となっています。



## 2. $H$ 上の作用 $A=|t_a|$

可換化すると

$$
X\mapsto X,\qquad
Y\mapsto Y+X,\qquad
Z\mapsto Z,\qquad
W\mapsto W.
$$

したがって、列を $X,Y,Z,W$ の像とする convention では

$$
A=
\begin{pmatrix}
1&1&0&0\\
0&1&0&0\\
0&0&1&0\\
0&0&0&1
\end{pmatrix}.
$$


In [5]:
H_BASIS = ('X', 'Y', 'Z', 'W')

def homology(element):
    v = vector(ZZ, 4)
    for idx in element.Tietze():
        i = abs(idx) - 1
        v[i] += 1 if idx > 0 else -1
    return v


def matrix_from_columns(columns, ring=ZZ):
    return matrix(
        ring,
        len(columns[0]),
        len(columns),
        lambda i, j: columns[j][i]
    )


A = matrix_from_columns(
    [homology(g) for g in images_a],
    ZZ
)

A

[1 1 0 0]
[0 1 0 0]
[0 0 1 0]
[0 0 0 1]

In [8]:
expected_A = matrix(ZZ, [
    [1, 1, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 1, 0],
    [0, 0, 0, 1],
])

assert A == expected_A

J = matrix(ZZ, [
    [ 0,  1, 0, 0],
    [-1,  0, 0, 0],
    [ 0,  0, 0, 1],
    [ 0,  0,-1, 0],
])

assert A.transpose() * J * A == J

print("A is symplectic.")

A is symplectic.



この $A$ は

$$
U\longmapsto U+(X\cdot U)X
$$

という $X=|a|$ に沿う transvection になっています。  
特に $X\cdot Y=1$ なので $Y\mapsto Y+X$ です。



## 3. $\ell_2$ の準備

現在採用している symplectic expansion の次数2部分は、生成元上で

$$
\ell_2(x)=\frac12X\wedge Y,\qquad
\ell_2(y)=-\frac12X\wedge Y,
$$

$$
\ell_2(z)=\frac12Z\wedge W,\qquad
\ell_2(w)=-\frac12Z\wedge W.
$$

また積に対して

$$
\ell_2(uv)
=
\ell_2(u)+\ell_2(v)
+\frac12|u|\wedge|v|
$$

を使います。


In [9]:
WEDGE_BASIS = (
    'X∧Y', 'X∧Z', 'X∧W',
    'Y∧Z', 'Y∧W', 'Z∧W'
)

WEDGE_PAIRS = (
    (0,1), (0,2), (0,3),
    (1,2), (1,3), (2,3)
)

def wedge(u, v):
    u = vector(QQ, u)
    v = vector(QQ, v)

    return vector(QQ, [
        u[i]*v[j] - u[j]*v[i]
        for i, j in WEDGE_PAIRS
    ])


def ell2_generators():
    return {
        1: vector(QQ, [ QQ(1)/2, 0, 0, 0, 0, 0]),  # x
        2: vector(QQ, [-QQ(1)/2, 0, 0, 0, 0, 0]),  # y
        3: vector(QQ, [0, 0, 0, 0, 0,  QQ(1)/2]),  # z
        4: vector(QQ, [0, 0, 0, 0, 0, -QQ(1)/2]),  # w
    }


def ell2_letter(idx, ell2_gen):
    value = vector(QQ, ell2_gen[abs(idx)])
    return value if idx > 0 else -value


def ell2(element, ell2_gen):
    h = vector(QQ, 4)
    ell = vector(QQ, 6)

    for idx in element.Tietze():
        v = vector(QQ, 4)
        i = abs(idx) - 1
        v[i] = 1 if idx > 0 else -1

        ell += (
            ell2_letter(idx, ell2_gen)
            + QQ(1)/2 * wedge(h, v)
        )

        h += v

    return ell


ELL2 = ell2_generators()


### 3.1 手計算で $\ell_2(yx)$ を確認

$t_a(y)=yx$ なので、この値が Route B の核心です。

$$
\begin{aligned}
\ell_2(yx)
&=\ell_2(y)+\ell_2(x)
  +\frac12\,Y\wedge X\\
&=-\frac12X\wedge Y
  +\frac12X\wedge Y
  -\frac12X\wedge Y\\
&=-\frac12X\wedge Y.
\end{aligned}
$$

したがって

$$
\ell_2(t_a(y))=\ell_2(y).
$$


In [10]:
ell_y  = ell2(fg.y, ELL2)
ell_x  = ell2(fg.x, ELL2)
ell_yx = ell2(fg.y * fg.x, ELL2)

print("ell2(y)  =", ell_y)
print("ell2(x)  =", ell_x)
print("ell2(yx) =", ell_yx)

assert ell_yx == ell_y

ell2(y)  = (-1/2, 0, 0, 0, 0, 0)
ell2(x)  = (1/2, 0, 0, 0, 0, 0)
ell2(yx) = (-1/2, 0, 0, 0, 0, 0)



## 4. Route B：$\pi$ への作用から $\tau_1^\theta(t_a)$ を求める

一般に

$$
\ell_2(\phi(g))
=
\tau_1^\theta(\phi)(A|g|)
+(\Lambda^2A)\ell_2(g)
$$

です。ここで $A=|\phi|$。

そこで

$$
\Delta_g
:=
\ell_2(\phi(g))
-(\Lambda^2A)\ell_2(g)
$$

と置くと

$$
\Delta_g=\tau_1^\theta(\phi)(A|g|).
$$

$g=x,y,z,w$ に対する $\Delta_g$ を列に並べた行列を $D$ とすれば

$$
D=\tau_1^\theta(\phi)A,
\qquad
\tau_1^\theta(\phi)=DA^{-1}.
$$


In [11]:
def wedge_action_matrix(A):
    A = A.change_ring(QQ)

    columns = []
    for i, j in WEDGE_PAIRS:
        columns.append(
            wedge(A.column(i), A.column(j))
        )

    return matrix_from_columns(columns, QQ)


A2 = wedge_action_matrix(A)

print("Lambda^2 A =")
print(A2)

Lambda^2 A =
[1 0 0 0 0 0]
[0 1 0 1 0 0]
[0 0 1 0 1 0]
[0 0 0 1 0 0]
[0 0 0 0 1 0]
[0 0 0 0 0 1]



$A(X)=X,\ A(Y)=X+Y$ なので、特に

$$
(\Lambda^2A)(X\wedge Y)
=
X\wedge(X+Y)
=
X\wedge Y.
$$

したがって $\ell_2(x)$ と $\ell_2(y)$ は $\Lambda^2A$ で不変です。


In [12]:
delta_columns = []

for name, g, image in zip(
    ('x','y','z','w'),
    fg.BASIS,
    images_a
):
    delta = (
        ell2(image, ELL2)
        - A2 * ell2(g, ELL2)
    )

    delta_columns.append(delta)
    print(f"Delta_{name} =", delta)

D = matrix_from_columns(delta_columns, QQ)

print()
print("D =")
print(D)

Delta_x = (0, 0, 0, 0, 0, 0)
Delta_y = (0, 0, 0, 0, 0, 0)
Delta_z = (0, 0, 0, 0, 0, 0)
Delta_w = (0, 0, 0, 0, 0, 0)

D =
[0 0 0 0]
[0 0 0 0]
[0 0 0 0]
[0 0 0 0]
[0 0 0 0]
[0 0 0 0]



4本すべてについて $\Delta_g=0$ になるので、

$$
D=0.
$$

したがって

$$
\boxed{\tau_1^\theta(t_a)=DA^{-1}=0}.
$$


In [13]:
tau_from_action = (
    D * A.change_ring(QQ).inverse()
)

print("tau_1^theta(t_a) from action on pi =")
print(tau_from_action)

assert tau_from_action == zero_matrix(QQ, 6, 4)

tau_1^theta(t_a) from action on pi =
[0 0 0 0]
[0 0 0 0]
[0 0 0 0]
[0 0 0 0]
[0 0 0 0]
[0 0 0 0]



## 5. Route A：Dehn twist formula から求める

`kiyoh.pdf` の Dehn twist formula は

$$
\boxed{
\tau_1^\theta(t_c)
=
-|c|\wedge\ell_2(c)
}.
$$

$a$-curve は $a=x$ なので

$$
|a|=X,
\qquad
\ell_2(a)
=
\frac12X\wedge Y.
$$

したがって

$$
\begin{aligned}
\tau_1^\theta(t_a)
&=
-X\wedge
\left(\frac12X\wedge Y\right)\\
&=
-\frac12X\wedge X\wedge Y\\
&=0.
\end{aligned}
$$

ここでは同じ $X$ が2回現れるため、$\Lambda^3H$ の段階ですでに $0$ です。  
したがって $\operatorname{Hom}(H,\Lambda^2H)$ への埋め込みを具体的に計算する必要もありません。


In [14]:
# Lambda^3 H の基底順序:
# X∧Y∧Z, X∧Y∧W, X∧Z∧W, Y∧Z∧W

TRIPLE_PAIRS = (
    (0,1,2),
    (0,1,3),
    (0,2,3),
    (1,2,3),
)

def wedge_H_Lambda2(u, eta):
    # u ∈ H, eta ∈ Lambda^2 H に対して
    # u ∧ eta ∈ Lambda^3 H を4成分で返す。
    u = vector(QQ, u)
    eta = vector(QQ, eta)

    def eta_coeff(i, j):
        if i == j:
            return QQ(0)
        if i < j:
            return eta[WEDGE_PAIRS.index((i, j))]
        return -eta[WEDGE_PAIRS.index((j, i))]

    values = []

    for i, j, k in TRIPLE_PAIRS:
        values.append(
            u[i] * eta_coeff(j, k)
            - u[j] * eta_coeff(i, k)
            + u[k] * eta_coeff(i, j)
        )

    return vector(QQ, values)


a_H = vector(QQ, homology(fg.x))
ell_a = ell2(fg.x, ELL2)

tau_formula_Lambda3 = -wedge_H_Lambda2(
    a_H,
    ell_a
)

print("-|a| wedge ell2(a) in Lambda^3 H =")
print(tau_formula_Lambda3)

assert tau_formula_Lambda3 == vector(QQ, 4)

-|a| wedge ell2(a) in Lambda^3 H =
(0, 0, 0, 0)



したがって Dehn twist formula 側でも

$$
\boxed{\tau_1^\theta(t_a)=0}.
$$

零元の $\Lambda^3H\hookrightarrow\operatorname{Hom}(H,\Lambda^2H)$ での像はもちろん零行列です。


In [15]:
tau_from_formula = zero_matrix(QQ, 6, 4)

assert tau_from_formula == tau_from_action

print("Route A == Route B :", tau_from_formula == tau_from_action)

Route A == Route B : True



# 6. 結論

今回の $t_a$ については、二つの計算が一致しました。

### Route B：$\pi$ への作用から

$$
t_a:
\begin{cases}
x\mapsto x,\\
y\mapsto yx,\\
z\mapsto z,\\
w\mapsto w,
\end{cases}
$$

を使うと

$$
D=0
$$

となり、

$$
\tau_1^\theta(t_a)=DA^{-1}=0.
$$

### Route A：Dehn twist formula から

$$
a=x,\qquad
\ell_2(a)=\frac12X\wedge Y
$$

なので

$$
\tau_1^\theta(t_a)
=
-X\wedge\frac12X\wedge Y
=
0.
$$

したがって

$$
\boxed{
\tau_1^\theta(t_a)
\text{ from action on }\pi
=
\tau_1^\theta(t_a)
\text{ from Dehn twist formula}
=
0
}.
$$

---

## 次に $t_b$ を見るときの違い

$t_a$ は $|a|\wedge\ell_2(a)=0$ だったため非常に簡単でした。

$t_b$ では一般に

$$
-|b|\wedge\ell_2(b)
$$

が非零になり得るので、

$$
\Lambda^3H
\hookrightarrow
\operatorname{Hom}(H,\Lambda^2H)
$$

を具体的に行列へ変換する部分も必要になります。  
その意味で、この $t_a$ の notebook は「最初の一例」として計算の骨格だけを追うのに向いています。
